# Part 1: Housekeeping and obtaininng ICEYE image

In [ ]:
my_gdrive_folder = 'drive/MyDrive/watchduty/'
#my_gee_folder = 'users/mickymags/watchduty/'

In [ ]:
!pip install ipyleaflet==0.18.2 geemap hydrafloods     # Install hydrafloods and its relevant dependencies
!pip install geemap

INFO: pip is looking at multiple versions of geemap to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of geemap to determine which version is compatible with other requirements. This could take a while.
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.6/86.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.0/70.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 43.4 MB/s eta 0:00:00
  Created wheel for pipetools: filename=pipetools-1.1.0-py3-none-any.whl size=13600 sha256=1f283ebce2a4ed178dabf1421d35474

In [ ]:
from hydrafloods import corrections
import hydrafloods as hf
import ee
import geemap
import google.colab
from osgeo import gdal
import numpy as np

In [ ]:
ee.Authenticate()
ee.Initialize(project = 'servir-sco-assets')

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_7TDKVSyKvBdmMqW?ref=4i2o6


In [ ]:
from google.colab import drive

In [ ]:
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
import os
import shutil

In [ ]:
ls

drive/  sample_data/


In [ ]:
os.chdir(my_gdrive_folder)

In [ ]:
ls

 ICEYE_DATA/                        PIPECAST_hydrafloods_prep
 louisiana_20200702.tif             PIPECAST_inital_sprint.ipynb
 PIPECAST_Dev_Update_8_18.gslides  'SPRINT_09 09 25.gdoc'


In [ ]:
in_file = 'louisiana_20200702.tif'
out_file = 'EDIT_louisiana_20200702.tif'

In [ ]:
ls

 ICEYE_DATA/                        PIPECAST_hydrafloods_prep
 louisiana_20200702.tif             PIPECAST_inital_sprint.ipynb
 PIPECAST_Dev_Update_8_18.gslides  'SPRINT_09 09 25.gdoc'


In [ ]:
cd ICEYE_DATA

/content/drive/MyDrive/watchduty/ICEYE_DATA


In [ ]:
ls

alabama_20230721.tif     california_v2_20230118.tif  louisiana_20200702.tif
california_20230118.tif  colorado_20220921.tif


In [ ]:
shutil.copy('louisiana_20200702.tif', 'edited_louisiana_20200702.tif')

'edited_louisiana_20200702.tif'

In [ ]:
in_file = 'edited_louisiana_20200702.tif'
out_file = 'EDIT2_louisiana_20200702.tif'

In [ ]:
ls

alabama_20230721.tif        colorado_20220921.tif
california_20230118.tif     edited_louisiana_20200702.tif
california_v2_20230118.tif  louisiana_20200702.tif


In [ ]:
!gdal_edit.py -a_srs EPSG:4326 edited_louisiana_20200702.tif

In [ ]:
!gdal_edit.py -a_ullr -91.65 30.175 -90.986 30.853 edited_louisiana_20200702.tif

Warning 1: edited_louisiana_20200702.tif: GCPs previously set are going to be cleared due to the setting of a geotransform.


In [ ]:
gdal.Warp('ULT_louisiana_20200702.tif', 'edited_louisiana_20200702.tif', dstSRS='EPSG:4326')

<osgeo.gdal.Dataset; proxy of <Swig Object of type 'GDALDatasetShadow *' at 0x7ece5beb7f60> >

In [ ]:
ls

alabama_20230721.tif        edited_louisiana_20200702.tif
california_20230118.tif     louisiana_20200702.tif
california_v2_20230118.tif  ULT_louisiana_20200702.tif
colorado_20220921.tif


In [ ]:
pwd

'/content/drive/MyDrive/watchduty/ICEYE_DATA'

In [ ]:
info = gdal.Info('ULT_louisiana_20200702.tif')

In [ ]:
inc_angle = info[info.find('INCIDENCE_CENTER'):info.find('INCIDENCE_CENTER')+30].split('=')[1]

'24.0633464461'

In [ ]:
new_ds = gdal.Open('ULT_louisiana_20200702.tif', gdal.GA_ReadOnly)

In [ ]:
geotransform = new_ds.GetGeoTransform()
proj = new_ds.GetProjection()
xsize = new_ds.RasterXSize
ysize = new_ds.RasterYSize
nbands = new_ds.RasterCount

# Create output dataset with one extra band
driver = gdal.GetDriverByName("GTiff")
out_ds = driver.Create('louisiana_20200702_1525.tif', xsize, ysize, nbands + 1, gdal.GDT_Float32)

# Copy georeferencing
out_ds.SetGeoTransform(geotransform)
out_ds.SetProjection(proj)

# Copy Original Band
in_band = new_ds.GetRasterBand(1)
data = in_band.ReadAsArray()
out_band = out_ds.GetRasterBand(1)
out_band.WriteArray(data)
out_band.SetDescription("VV")

# Preserve NoData and description if available
nodata = in_band.GetNoDataValue()
if nodata is not None:
  out_band.SetNoDataValue(nodata)

desc = in_band.GetDescription()
if desc:
  out_band.SetDescription(desc)

# Create constant incidence angle band
const_array = np.full((ysize, xsize), inc_angle, dtype = np.float32)
angle_band = out_ds.GetRasterBand(2)
angle_band.WriteArray(const_array)
angle_band.SetDescription("angle")

# Save and close
out_ds.FlushCache()
out_ds = None
src_ds = None

In [ ]:
ls l*

louisiana_20200702_1525.tif  louisiana_20200702.tif


# Upload dataset to GEE

In [ ]:
xsize

22441

# Arkansas Image

In [ ]:
pwd

'/content/drive/MyDrive/watchduty'

In [ ]:
ls

 ICEYE_DATA/                        PIPECAST_hydrafloods_prep
 louisiana_20200702.tif             PIPECAST_inital_sprint.ipynb
 PIPECAST_Dev_Update_8_18.gslides  'SPRINT_09 09 25.gdoc'


In [ ]:
cd ICEYE_DATA

[Errno 2] No such file or directory: 'ICEYE_DATA'
/content/drive/MyDrive/watchduty/ICEYE_DATA


In [ ]:
shutil.copy('arkansas_20200331.tif', 'arkansas_edit_0918.tif')

'arkansas_edit_0918.tif'

In [ ]:
ls ark*

arkansas_20200331_1608.tif  arkansas_edit_0918.tif   arkansas_trying.tif
arkansas_20200331.tif       arkansas_FiNaL_0918.tif
arkansas_ayall.tif          arkansas_sandbox.tif


In [ ]:
shutil.copy('arkansas_20200331.tif', 'arkansas_take3.tif')

'arkansas_take3.tif'

In [ ]:
ls ark*

arkansas_20200331_1608.tif  arkansas_edit_0918.tif   arkansas_take3.tif
arkansas_20200331.tif       arkansas_FiNaL_0918.tif  arkansas_trying.tif
arkansas_ayall.tif          arkansas_sandbox.tif


In [ ]:
!gdal_edit.py -a_srs EPSG:4326 arkansas_take3.tif

In [ ]:
!gdal_edit.py -a_ullr -89.767 35.481 -90.47 34.805 arkansas_take3.tif

Warning 1: arkansas_take3.tif: GCPs previously set are going to be cleared due to the setting of a geotransform.


In [ ]:
ls arkansas_ta*

arkansas_take3.tif


In [ ]:
shutil.copy('arkansas_take3.tif', 'arkansas_take3_final.tif')

'arkansas_take3_final.tif'

In [ ]:
ls arkansas_take3*

arkansas_take3_final.tif  arkansas_take3.tif


In [ ]:
import time

In [ ]:
time.sleep(60)

In [ ]:
shutil.copy('arkansas_20200331.tif', 'arkansas_trying.tif')

'arkansas_trying.tif'

In [ ]:
!gdal_edit.py -a_srs EPSG:4326 arkansas_trying.tif

In [ ]:
!gdal_edit.py -a_ullr -89.767 35.481 -90.47 34.805 arkansas_trying.tif

Warning 1: arkansas_trying.tif: GCPs previously set are going to be cleared due to the setting of a geotransform.


In [ ]:
arkansas_info = gdal.Info('arkansas_take3.tif')

In [ ]:
ark_inc_angle = arkansas_info[arkansas_info.find('INCIDENCE_CENTER'):arkansas_info.find('INCIDENCE_CENTER')+30].split('=')[1]
ark_inc_angle

'19.6246748293'

In [ ]:
art3_ds = gdal.Open('arkansas_take3.tif', gdal.GA_ReadOnly)

In [ ]:
art3_ds

<osgeo.gdal.Dataset; proxy of <Swig Object of type 'GDALDatasetShadow *' at 0x7b68eb7d9650> >

In [ ]:
geotransform = art3_ds.GetGeoTransform()
proj = art3_ds.GetProjection()
xsize = art3_ds.RasterXSize
ysize = art3_ds.RasterYSize
nbands = art3_ds.RasterCount

# Create output dataset with one extra band
driver = gdal.GetDriverByName("GTiff")
out_ds = driver.Create('arkansas_take3.tif', xsize, ysize, nbands + 1, gdal.GDT_Float32)

# Copy georeferencing
out_ds.SetGeoTransform(geotransform)
out_ds.SetProjection(proj)

# Copy Original Band
in_band = art3_ds.GetRasterBand(1)
data = in_band.ReadAsArray()
out_band = out_ds.GetRasterBand(1)
out_band.WriteArray(data)
out_band.SetDescription("VV")

# Preserve Nodata and description if available
nodata = in_band.GetNoDataValue()
if nodata is not None:
  out_band.SetNoDataValue(nodata)

desc = in_band.GetDescription()
if desc:
  out_band.SetDescription(desc)

# Create constant incidence angle band
const_array = np.full((ysize, xsize), ark_inc_angle, dtype = np.float32)
angle_band = out_ds.GetRasterBand(2)
angle_band.WriteArray(const_array)
angle_band.SetDescription("angle")

# Save and close
out_ds.FlushCache()
out_ds = None
src_ds = None

In [ ]:
shutil.copy('arkansas_take3.tif', 'arkansas_take3_update.tif')

'arkansas_take3_update.tif'

In [ ]:
!gdal_edit.py -a_ullr -90.47 35.481 -89.767 34.805 arkansas_edit_0918.tif

In [ ]:
!gdal_edit.py -a_srs EPSG:4326 arkansas_sandbox.tif

In [ ]:
ls ark*

arkansas_20200331_1608.tif  arkansas_20200331.tif  arkansas_edit_0918.tif


In [ ]:
in_file = 'edited_arkansas_20200331.tif'
out_file = 'inter_arkansas_20200331.tif'

In [ ]:
!gdal_edit.py -a_srs EPSG:4326 arkansas_edit_0918.tif

In [ ]:
!gdal_edit.py -a_ullr -90.47 35.481 -89.767 34.805 arkansas_edit_0918.tif

Warning 1: arkansas_edit_0918.tif: GCPs previously set are going to be cleared due to the setting of a geotransform.


In [ ]:
gdal.Warp('arkansas_FiNaL_0918.tif', 'arkansas_edit_0918.tif', dstSRS='EPSG:4326')

/usr/local/lib/python3.12/dist-packages/osgeo/gdal.py:312: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


<osgeo.gdal.Dataset; proxy of <Swig Object of type 'GDALDatasetShadow *' at 0x7aeb39ea7420> >

In [ ]:
ls ark*

arkansas_20200331_1608.tif  arkansas_edit_0918.tif
arkansas_20200331.tif       arkansas_FiNaL_0918.tif


In [ ]:
arkansas_info = gdal.Info('arkansas_FiNaL_0918.tif')

In [ ]:
inc_angle = arkansas_info[arkansas_info.find('INCIDENCE_CENTER'):arkansas_info.find('INCIDENCE_CENTER')+30].split('=')[1]

In [ ]:
inc_angle

'19.6246748293'

In [ ]:
ar_ds = gdal.Open('arkansas_FiNaL_0918.tif', gdal.GA_ReadOnly)

In [ ]:
ar_ds = gdal.Open('arkansas_FiNaL_0918.tif', gdal.GA_ReadOnly)

In [ ]:
ls ark*

arkansas_20200331_1608.tif  arkansas_edit_0918.tif
arkansas_20200331.tif       arkansas_FiNaL_0918.tif


In [ ]:
gdal.Warp('semifinal_arkansas_0918.tif', 'arkansas_FiNaL_0918.tif', dstSRS='EPSG:4326')

# Alabama Image

In [ ]:
ls

alabama_20230721.tif          edited_louisiana_20200702.tif
arkansas_20200331_1608.tif    fin_louisiana_20200702.tif
arkansas_20200331.tif         louisiana_20200702_1525.tif
california_20230118.tif       louisiana_20200702.tif
california_v2_20230118.tif    ULT_arkansas_20200331.tif
colorado_20220921.tif         ULT_louisiana_20200702.tif
edited_arkansas_20200331.tif


In [ ]:
shutil.copy('alabama_20230721.tif', 'edited_alabama_20230721.tif')

'edited_alabama_20230721.tif'

In [ ]:
in_file = 'edited_alabama_20230721.tif'
out_file = 'EDIT2_alabama_20230721.tif'

In [ ]:
ls

alabama_20230721.tif         edited_arkansas_20200331.tif
arkansas_20200331_1608.tif   edited_louisiana_20200702.tif
arkansas_20200331.tif        fin_louisiana_20200702.tif
california_20230118.tif      louisiana_20200702_1525.tif
california_v2_20230118.tif   louisiana_20200702.tif
colorado_20220921.tif        ULT_arkansas_20200331.tif
edited_alabama_20230721.tif  ULT_louisiana_20200702.tif


In [ ]:
!gdal_edit.py -a_srs EPSG:4326 edited_alabama_20230721.tif

In [ ]:
!gdal_edit.py -a_ullr -86.976 34.367 -86.485 35.174 edited_alabama_20230721.tif

Warning 1: edited_alabama_20230721.tif: GCPs previously set are going to be cleared due to the setting of a geotransform.


In [ ]:
gdal.Warp('ULT_alabama_20230721.tif', 'edited_alabama_20230721.tif', dstSRS='EPSG:4326')

<osgeo.gdal.Dataset; proxy of <Swig Object of type 'GDALDatasetShadow *' at 0x7ece5bd98180> >

In [ ]:
inc_angle = info[info.find('INCIDENCE_CENTER'):info.find('INCIDENCE_CENTER')+30].split('=')[1]

In [ ]:
al_ds = gdal.Open('ULT_alabama_20230721.tif', gdal.GA_ReadOnly)

In [ ]:
geotransform = al_ds.GetGeoTransform()
proj = al_ds.GetProjection()
xsize = al_ds.RasterXSize
ysize = al_ds.RasterYSize
nbands = al_ds.RasterCount

# Create output dataset with one extra band
driver = gdal.GetDriverByName("GTiff")
out_ds = driver.Create('alabama_20230721_1649.tif', xsize, ysize, nbands + 1, gdal.GDT_Float32)

# Copy georeferencing
out_ds.SetGeoTransform(geotransform)
out_ds.SetProjection(proj)

# Copy Original Band
in_band = al_ds.GetRasterBand(1)
data = in_band.ReadAsArray()
out_band = out_ds.GetRasterBand(1)
out_band.WriteArray(data)
out_band.SetDescription("VV")

# Preserve Nodata and description if available
nodata = in_band.GetNoDataValue()
if nodata is not None:
  out_band.SetNoDataValue(nodata)

desc = in_band.GetDescription()
if desc:
  out_band.SetDescription(desc)

# Create constant incidence angle band
const_array = np.full((ysize, xsize), inc_angle, dtype = np.float32)
angle_band = out_ds.GetRasterBand(2)
angle_band.WriteArray(const_array)
angle_band.SetDescription("angle")

# Save and close
out_ds.FlushCache()
out_ds = None
src_ds = None